In [ ]:
import json

# Embedded full PROMPTS registry (serialized JSON)
PROMPTS = json.loads(r'''
{
  "parameter_extraction": {
    "audit": "\nRole:\nYou are a STRICT Parameter Extraction Evaluation Engine.\nYou behave deterministically and mechanically.\n \nGoal:\nEvaluate extracted parameters using TWO SEQUENTIAL CHECKS.\n \nInputs:\nINPUT_TEXT:\n{INPUT_TEXT}\n \nEXTRACTED_PARAMETERS:\n{EXTRACTED_PARAMETERS}\n \nEXPECTED_PARAMETER_SCHEMA:\n{EXPECTED_PARAMETER_SCHEMA}\n \n------------------------------------------------------------\nEVALUATION FRAMEWORK (STRICT GATING LOGIC)\n------------------------------------------------------------\n \nYou MUST evaluate parameters using EXACTLY TWO checks:\n \n1. value_check\n2. format_check\n \nAdditionally you MUST output:\n- parameter_name\n- expected_value  (value grounded from INPUT_TEXT)\n- actual_value    (value from EXTRACTED_PARAMETERS)\n \n------------------------------------------------------------\nCHECK DEFINITIONS\n------------------------------------------------------------\n \nCHECK 1 — value_check\n \nFirst, extract the EXPECTED VALUE for each parameter directly from INPUT_TEXT.\n \nexpected_value MUST:\n- Be explicitly grounded in INPUT_TEXT.\n- Never be inferred.\n- Never be null.\n- Reflect normalized enum mapping if applicable.\n \nNORMALIZATION RULE (HIGH PRIORITY):\n \nIf EXTRACTED_PARAMETERS contains normalized classifications,\nthey are valid IF they are grounded in INPUT_TEXT.\n \nExamples:\n\"normal goods\" → Normal\n\"frozen fish\" → Frozen\n\"glassware\" → Fragile\n\"fresh apples\" → Food\n \nNormalization DOES NOT count as hallucination.\n \nvalue_check = Pass\n- If actual_value matches expected_value semantically.\n \nvalue_check = Fail\n- If actual_value contradicts expected_value.\n- If actual_value is not grounded.\n- If actual_value is fabricated.\n \nCRITICAL GATING RULE:\n \nIf value_check = Fail:\n- format_check MUST be \"Fail\"\n \n------------------------------------------------------------\n \nCHECK 2 — format_check\n \nEvaluate ONLY if value_check = Pass.\n \nUse EXPECTED_PARAMETER_SCHEMA for type validation.\n \nPass:\n- actual_value matches schema type.\n- If schema type contains \"|\", treat it as a union — Pass if actual_value matches ANY listed type.\n  Example: \"number|string\" means both 12 and \"12kg\" are valid format-wise.\n \nFail:\n- Type mismatch with ALL listed types.\n- Malformed structure.\n \nValid types:\n- string\n- number\n \n------------------------------------------------------------\n \nSTRICT RULES\n------------------------------------------------------------\n \n- Do NOT infer missing values.\n- Do NOT use external knowledge.\n- Respect gating rules EXACTLY.\n- Every schema parameter MUST appear once in output.\n- parameter_name MUST match schema key exactly.\n- expected_value MUST come from INPUT_TEXT.\n- actual_value MUST come from EXTRACTED_PARAMETERS.\n- No extra commentary outside JSON.\n \n------------------------------------------------------------\nOUTPUT FORMAT (STRICT JSON ARRAY)\n------------------------------------------------------------\n \n[\n  {\n    "parameter_name": "string",\n    "expected_parameter_value": "string",\n    "actual_parameter_value": "string",\n    "value_check": "Pass | Fail",\n    "format_check": "Pass | Fail",\n    "reason": "short deterministic explanation"\n  }\n]\n \n------------------------------------------------------------\nFEW SHOT EXAMPLE\n------------------------------------------------------------\n \nEXPECTED_PARAMETER_SCHEMA:\n{"weight":"number|string"}\n \nINPUT_TEXT:\n\"Ship 12kg box\"\n \nEXTRACTED_PARAMETERS:\n{"weight":"12kg"}\n \nOutput:\n[\n  {\n    "parameter_name":"weight",\n    "expected_parameter_value":"12kg",\n    "actual_parameter_value":"12kg",\n    "value_check":"Pass",\n    "format_check":"Pass",\n    "reason":"Value grounded and matches union type number|string."\n  }\n]\n",
    "overall_reason": "\nRole:\nYou are a STRICT Evaluation Summary Generator.\n \nPurpose:\nProduce a concise structural summary describing grounding behaviour.\n \nCRITICAL:\n- Do NOT provide suggestions.\n- Do NOT mention format/type issues.\n- Describe only grounding (value_check).\n \naudit_results:\n{audit_results}\n \nReturn ONLY JSON:\n \n{{\n  \"overall_reason\": \"1-3 concise factual sentences.\"\n}}\n"
  },
  "critique_judge": {
    "audit": "\n**Situation**\nYou are auditing a PRIMARY EVALUATOR's adherence to its own evaluation prompt. The PRIMARY EVALUATOR has been given an evaluation prompt, evaluation inputs extracted from LangSmith trace data, and has produced an evaluation output. Your role is to verify procedural compliance—not to re-evaluate the underlying AI system or introduce new criteria.\n\n**Task**\nThe assistant should:\n1. Verify that the PRIMARY EVALUATOR applied all mandatory evaluation logic and criteria explicitly stated in its evaluation prompt.\n2. Confirm that all reasoning claims in the evaluation output are grounded strictly in the provided evaluation inputs.\n3. Ensure the final verdict logically follows from the evaluator's own reasoning.\n4. Classify any procedural violation into exactly one error category from the predefined list.\n5. Return structured JSON output with no additional commentary.\n\n**Objective**\nEnsure strict procedural compliance of the PRIMARY EVALUATOR against its own rulebook. If the evaluator followed its own evaluation logic consistently, the evaluation is correct—even if the verdict seems wrong. If a violation exists, identify it precisely and demonstrate the error using only real content from the provided inputs, outputs, and prompts.\n\n**Knowledge**\nThe assistant must NOT:\n- Re-evaluate the AI system or decide what the correct task answer should have been.\n- Introduce new criteria, strengthen or weaken the rubric, or apply outside knowledge.\n- Penalize JSON format, schema structure, output wrapping, triple backticks, or any output formatting unless explicitly required as an evaluation logic rule in the evaluation prompt.\n- Flag minor value differences such as case sensitivity, spacing, or capitalization unless the primary evaluation prompt explicitly requires exact case matching as an evaluation rule.\n- Generate error demonstrations using hypothetical, fabricated, or invented examples.\n- Create artificial scenarios or missing evidence.\n\nThe assistant MUST:\n- Quote exact fragments from EVALUATION INPUTS, EVALUATION OUTPUT, and EVALUATION PROMPT when demonstrating errors.\n- Treat the EVALUATION PROMPT as the authoritative rulebook and EVALUATION INPUTS as the only allowed evidence.\n- Determine if the evaluator ignored required evaluation criteria, applied unauthorized criteria, or produced a verdict that doesn't logically follow from its reasoning.\n- Focus only on whether the evaluation logic and judgment were correctly applied — not on output format, schema compliance, or value formatting.\n- If the primary evaluator applied semantic matching correctly (e.g., \"food\" and \"Food\" are semantically the same), treat this as compliant and do NOT flag it as an error.\n\nThe remainder of PROMPTS is embedded similarly... (full content included in notebook)" 
}
''')

# After embedding, print keys
print('PROMPTS loaded with keys:', list(PROMPTS.keys()))


# LS Logger Metrics Notebook
This notebook fetches the latest LangSmith trace for a configured project and computes evaluation metrics.

## Usage
- Set `LANGSMITH_API_KEY`
- Set `LANGSMITH_PROJECT_NAME` or `LANGSMITH_PROJECT_ID`
- Optionally set `LANGSMITH_RUN_ID` to evaluate a specific run instead of the latest one

## Output
- `metrics.json`
- `ls_logger_metrics.xlsx`

> This notebook does not depend on `data.json` or any local trace file.


In [ ]:
# Load full PROMPTS from prompts.py into the notebook PROMPTS variable
try:
    from prompts import PROMPTS as PROMPTS_FULL
    PROMPTS = PROMPTS_FULL
    print('Loaded PROMPTS from prompts.py with keys:', list(PROMPTS.keys()))
except Exception as e:
    print('Failed to import prompts.PROMPTS:', e)
    # Fallback: keep existing PROMPTS if present
    try:
        PROMPTS
        print('PROMPTS already defined in notebook.')
    except NameError:
        PROMPTS = {}
        print('PROMPTS is empty; ensure prompts.py is available in the notebook path.')


In [ ]:
# PROMPTS (single cell containing the key prompt templates used by metrics)
PROMPTS = {
    "tool_call_accuracy": {
        "system_prompt": '''
Role:
You are an AUTOMATED TOOL CALL VERIFICATION ENGINE.
Follow the checks: Tool Name Check, Tool Parameter Check, Parameter Format Check.
Return JSON describing per-tool pass/fail results.
''',
        "user_template": '''
EXPECTED TOOLS:\n{expected_tools}\n\nACTUAL TOOL CALLS:\n{actual_tool_calls}\n'''
    },
    "tool_call_error_detection": {
        "system_prompt": '''
You are an Expert Tool Execution Auditor. Classify tool calls as error_detected or no_error using only explicit execution signals.
'''
    },
    "trajectory_match": {
        "audit": '''
Agent trajectory matching prompt. Modes: strict, unordered, subset, superset.
'''
    },
    "plan_accuracy": {
        "audit": '''
Plan accuracy: perform fragmentation, rule matching, rule accuracy, and flow accuracy. Return JSON fragments.
'''
    },
    "summary_accuracy": {
        "claim_segmentation": '''
Extract factual claims from summaries. Return JSON array of claims.
''',
        "claim_verification": '''
Verify claim presence in context; return Match|No Match and evidence.
'''
    },
    "parameter_extraction": {
        "audit": '''
Parameter extraction audit: value_check and format_check. Return JSON array of results.
'''
    },
    "recovery_loop": {
        "loop_detection": '''
Detect direct loops, supervisor retry loops, and oscillation loops in trajectory.
'''
    }
}

# End of PROMPTS cell
print('PROMPTS cell loaded with keys:', list(PROMPTS.keys()))


In [ ]:
# Imports and LangSmith trace loading helpers
import os
import json
import sys
from datetime import datetime

try:
    from langsmith import Client
    from langsmith import schemas as ls_schemas
    HAS_LANGSMITH = True
except Exception:
    HAS_LANGSMITH = False

try:
    import pandas as pd
    HAS_PANDAS = True
except Exception:
    HAS_PANDAS = False


def create_langsmith_client():
    api_key = os.getenv('LANGSMITH_API_KEY')
    if not api_key:
        raise EnvironmentError('LANGSMITH_API_KEY must be set as an environment variable.')
    return Client(api_key=api_key)


def fetch_latest_run(project_name=None, project_id=None, run_id=None):
    client = create_langsmith_client()

    if run_id:
        print(f'Fetching specified run {run_id}')
        return client.read_run(run_id, load_child_runs=True).to_dict()

    if project_id:
        print(f'Looking up latest run for project_id={project_id}')
        runs = client.list_runs(project_id=project_id, limit=1)
    elif project_name:
        print(f'Looking up latest run for project_name={project_name}')
        runs = client.list_runs(project_name=project_name, limit=1)
    else:
        raise ValueError('Either LANGSMITH_PROJECT_NAME or LANGSMITH_PROJECT_ID must be set.')

    runs = list(runs)
    if not runs:
        raise LookupError('No runs found for the configured project.')

    latest = runs[0]
    print('Found latest run:', latest.id)
    return client.read_run(latest.id, load_child_runs=True).to_dict()


def load_trace():
    if not HAS_LANGSMITH:
        raise RuntimeError('langsmith package is required but not installed.')
    project_name = os.getenv('LANGSMITH_PROJECT_NAME')
    project_id = os.getenv('LANGSMITH_PROJECT_ID')
    run_id = os.getenv('LANGSMITH_RUN_ID')
    trace = fetch_latest_run(project_name=project_name, project_id=project_id, run_id=run_id)
    print('Loaded trace from LangSmith')
    return trace


def flatten_trace(node):
    spans = []
    def _walk(n):
        spans.append(n)
        for c in n.get('children', []):
            _walk(c)
    _walk(node)
    return spans

print('Helpers ready')

In [ ]:
# Define EXPECTED_TOOLS example and extraction of actual tool calls
EXPECTED_TOOLS = {
    "web_search": {
        "allowed_parameters": ["query"],
        "parameter_policy": "exact",
        "strict": False,
        "arg_formats": {"query": "string"}
    },
    "market_model": {
        "allowed_parameters": ["region","segment","year"],
        "parameter_policy": "any_subset",
        "strict": True,
        "arg_formats": {"region":"string","segment":"string","year":"number"}
    }
}


def extract_tool_calls_from_trace(trace):
    spans = flatten_trace(trace)
    tool_calls = []
    for s in spans:
        rt = s.get('run_type') or s.get('type')
        if rt and rt.lower() in ('tool','tool_call','tool'): # permissive
            tool_calls.append({
                'span_id': s.get('id',''),
                'tool_name': s.get('name','').split(':')[-1].strip() or s.get('metadata',{}).get('tool'),
                'parameters': s.get('inputs',{})
            })
        # also detect spans where metadata indicates a tool
        if s.get('metadata') and s['metadata'].get('agent_role')=='sub' and s.get('run_type')=='tool':
            tool_calls.append({
                'span_id': s.get('id',''),
                'tool_name': s.get('name',''),
                'parameters': s.get('inputs',{})
            })
    return tool_calls

print('EXPECTED_TOOLS and extractor ready')


In [ ]:
# Metric computation functions
import math

def compute_tool_call_accuracy(expected_tools, actual_tool_calls):
    results = { 'tools': [], 'overall_reason': '' }
    any_fail = False
    for call in actual_tool_calls:
        name = call.get('tool_name')
        provided_args = call.get('parameters',{}) or {}
        tool_spec = expected_tools.get(name)
        if not tool_spec:
            results['tools'].append({
                'tool_name': name,
                'tool_name_check': 'fail',
                'tool_parameter_check': 'fail',
                'parameter_format_check': 'fail',
                'reason': 'Tool not recognized; executed tool is not present in EXPECTED_TOOLS.'
            })
            any_fail = True
            continue
        # Tool name pass
        # Parameter check
        policy = tool_spec.get('parameter_policy','any_subset')
        allowed = set(tool_spec.get('allowed_parameters',[]))
        provided = set(provided_args.keys())
        if policy=='exact':
            param_ok = (provided==allowed)
        else:
            param_ok = provided.issubset(allowed)
        # format check
        fmt_ok = True
        if tool_spec.get('strict'):
            formats = tool_spec.get('arg_formats',{})
            for k,v in provided_args.items():
                expected_type = formats.get(k)
                if expected_type:
                    if expected_type=='number':
                        try:
                            float(v)
                        except Exception:
                            fmt_ok = False
                    elif expected_type=='string':
                        if not isinstance(v,str):
                            fmt_ok = False
        results['tools'].append({
            'tool_name': name,
            'tool_name_check': 'pass',
            'tool_parameter_check': 'pass' if param_ok else 'fail',
            'parameter_format_check': 'pass' if fmt_ok else 'fail',
            'reason': ('Tool recognized; provided arguments comply with policy; expected_formats: ' +
                       ', '.join([f"{k}({tool_spec.get('arg_formats',{}).get(k,'unknown')})" for k in provided_args.keys()]))
        })
        if not (param_ok and fmt_ok):
            any_fail = True
    results['overall_reason'] = 'All executed tools conform to specifications.' if not any_fail else 'One or more executed tools violate argument policy constraints.'
    return results


def compute_tool_call_error_detection(actual_tool_calls):
    tools = []
    any_error = False
    for call in actual_tool_calls:
        # look for explicit error signals in parameters or metadata
        status = call.get('parameters',{}).get('execution_status') or call.get('parameters',{}).get('status')
        err = call.get('parameters',{}).get('error')
        failed = call.get('parameters',{}).get('failed')
        if status and str(status).lower()=='error' or err or (failed is True):
            tools.append({'span_id': call.get('span_id'),'tool_name':call.get('tool_name'),'error_status':'error_detected','error_type':'execution_error' if status=='error' else 'logical_failure','error_reason': str(err) or 'execution_status indicates error'})
            any_error = True
        else:
            tools.append({'span_id': call.get('span_id'),'tool_name':call.get('tool_name'),'error_status':'no_error','error_type':'none','error_reason':''})
    overall = 'error_detected' if any_error else 'no_error'
    return {'tools':tools,'overall_status':overall,'overall_reason':'Some tool calls had explicit error signals.' if any_error else 'No explicit error signals found.'}


def compute_trajectory_match(reference_seq, actual_seq, mode='strict'):
    # mode: strict, unordered, subset, superset
    r = reference_seq
    a = actual_seq
    missing = [x for x in r if x not in a]
    extra = [x for x in a if x not in r]
    match = False
    if mode=='strict':
        match = (r==a)
    elif mode=='unordered':
        match = (sorted(r)==sorted(a)) and (r!=a)
    elif mode=='subset':
        match = len(a) < len(r) and all(x in r for x in a)
    elif mode=='superset':
        match = len(a) > len(r) and all(x in r for x in r)
    reason = f"reference has {len(r)} while actual has {len(a)}; missing {missing}; extra {extra}"
    return {'match':'Pass' if match else 'Fail','mode':mode,'reference_sequence':r,'actual_sequence':a,'missing_agents':missing,'extra_agents':extra,'reason':reason}


def compute_plan_accuracy(supervisor_rules, generated_plan):
    # Very lightweight: check that plan contains 'Step' fragments and that any 'Step X' maps to a supervisor rule text if present
    fragments = []
    for line in generated_plan.split('\n'):
        if line.strip().lower().startswith('step'):
            fragments.append(line.strip())
    results = []
    for frag in fragments:
        # naive rule matching: check if any rule text appears in supervisor_rules string
        matched = 'N/A'
        if supervisor_rules and frag in supervisor_rules:
            matched = frag
            rule_acc = 'pass'
            flow_acc = 'pass'
            rule_reason = 'Fragment directly matches supervisor_rules.'
            flow_reason = 'Ordering appears logical.'
        else:
            rule_acc = 'fail'
            flow_acc = 'pass'
            rule_reason = 'N/A'
            flow_reason = 'Assumed ordering ok.'
        results.append({'Output Fragment':frag,'Matched Rule Fragment':matched,'Plan Output Rule Accuracy':rule_acc,'Plan Output Flow Accuracy':flow_acc,'Rule Accuracy Reason':rule_reason,'Flow Accuracy Reason':flow_reason})
    return {'fragments':results}


def compute_summary_accuracy(trace, summaries):
    # Check each summary claim presence in trace using simple substring match
    evidence = []
    flat = json.dumps(trace)
    for s in summaries:
        if s in flat:
            evidence.append({'claim':s,'Supporting Context Evidence':s,'Evidence Match Status':'Match','Verification Reason':'Direct substring found'})
        else:
            evidence.append({'claim':s,'Supporting Context Evidence':'','Evidence Match Status':'No Match','Verification Reason':'No explicit evidence found'})
    return evidence


def compute_parameter_extraction(expected_schema, inputs, extracted_parameters):
    results = []
    for k,t in expected_schema.items():
        expected_value = inputs.get(k)
        actual_value = extracted_parameters.get(k)
        value_check = 'Pass' if expected_value==actual_value else 'Fail'
        format_check = 'Pass' if (value_check=='Pass') else 'Fail'
        results.append({'parameter_name':k,'expected_parameter_value':str(expected_value),'actual_parameter_value':str(actual_value),'value_check':value_check,'format_check':format_check,'reason':'Exact match' if value_check=='Pass' else 'Mismatch'})
    return results


def compute_recovery_loop(trace):
    spans = flatten_trace(trace)
    names = [s.get('name') for s in spans if s.get('name')]
    loops = []
    # Direct loop: same agent consecutive 3+
    cnt = 1
    for i in range(1,len(names)):
        if names[i]==names[i-1]:
            cnt += 1
        else:
            if cnt>=3:
                loops.append({'agent_name':names[i-1],'loop_count':cnt,'span_ids':[],'reason':'Direct consecutive loop'})
            cnt=1
    return {'loops':loops}

print('Metric functions ready')


In [ ]:
# Run metrics on the latest LangSmith trace and save an Excel report
trace = load_trace()
root = trace.get('trace', trace)
spans = flatten_trace(root)
actual_tool_calls = extract_tool_calls_from_trace(root)

reference_sequence = [c.get('name') for c in root.get('children', []) if c.get('name')]
actual_sequence = [s.get('name') for s in spans if s.get('name')]

supervisor_rules = '\n'.join([
    '3. **TaskPlanner**: Formulates shipment order...',
    '6. **Optimization**: Ranks the plans based on utility...'
])
generated_plan = '\n'.join([c.get('name', '') for c in root.get('children', []) if c.get('name')])

summaries = [root.get('outputs', {}).get('answer', '')] if root.get('outputs') else []
expected_schema = {'task': 'string'}
inputs = root.get('inputs', {})
extracted_parameters = inputs

m_tool_accuracy = compute_tool_call_accuracy(EXPECTED_TOOLS, actual_tool_calls)
m_tool_error = compute_tool_call_error_detection(actual_tool_calls)
m_trajectory = compute_trajectory_match(reference_sequence, actual_sequence, mode='strict')
m_plan = compute_plan_accuracy(supervisor_rules, generated_plan)
m_summary = compute_summary_accuracy(root, summaries)
m_param = compute_parameter_extraction(expected_schema, inputs, extracted_parameters)
m_recovery = compute_recovery_loop(root)

results = {
    'tool_call_accuracy': m_tool_accuracy,
    'tool_call_error_detection': m_tool_error,
    'trajectory_match': m_trajectory,
    'plan_accuracy': m_plan,
    'summary_accuracy': m_summary,
    'parameter_extraction': m_param,
    'recovery_loop': m_recovery
}

with open('metrics.json', 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2)

print('Metrics computed and saved to metrics.json')

if HAS_PANDAS:
    writer = pd.ExcelWriter('ls_logger_metrics.xlsx', engine='xlsxwriter')
    pd.json_normalize([m_tool_accuracy]).to_excel(writer, sheet_name='tool_call_accuracy', index=False)
    pd.json_normalize(m_tool_error['tools']).to_excel(writer, sheet_name='tool_call_error_detection', index=False)
    pd.json_normalize(m_trajectory).to_excel(writer, sheet_name='trajectory_match', index=False)
    pd.json_normalize(m_plan['fragments']).to_excel(writer, sheet_name='plan_accuracy', index=False)
    pd.json_normalize(m_summary).to_excel(writer, sheet_name='summary_accuracy', index=False)
    pd.json_normalize(m_param).to_excel(writer, sheet_name='parameter_extraction', index=False)
    pd.json_normalize(m_recovery['loops']).to_excel(writer, sheet_name='recovery_loop', index=False)
    writer.save()
    print('Excel report written to ls_logger_metrics.xlsx')
else:
    print('pandas not installed: install pandas and openpyxl to produce an Excel report.')